<a href="https://colab.research.google.com/github/j22k/Malayalam-TTS/blob/main/Notebooks/Finetune_F5_TTS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tune Malayalam F5-TTS on Kathbath (Colab T4)

**Goal:** fine-tune the pretrained Malayalam F5-TTS model [`multilingual-tts/F5-TTS-OpenBible-Malayalam`](https://huggingface.co/multilingual-tts/F5-TTS-OpenBible-Malayalam) (F5TTS_v1_Base, 24 kHz, vocos, 119-character vocab) on the Kathbath dataset prepared in notebook 2.

**Pipeline position:** `1. EDA` → `2. Preprocess` → **`3. Finetune (this notebook)`**

### ⚠️ Tested environment: do not change these versions
| Component | Version |
|---|---|
| Colab runtime | **T4 GPU** (15 GB), 12.7 GB RAM, Python 3.13.15 |
| torch / torchaudio / torchvision | `2.8.0+cu128` / `2.8.0+cu128` / `0.23.0+cu128` |
| f5-tts | `1.1.22` (all script patches below target this exact version) |
| transformers | `5.16.1` |

### Problems in the first version of this notebook, and how they are fixed now
| # | Problem (observed) | Root cause (f5-tts 1.1.22 source) | Fix in this notebook |
|---|---|---|---|
| 1 | The run printed `Saved last checkpoint at update 124848` with **no training steps** | The HF checkpoint is a *full training state* (`update=124848`, optimizer, scheduler). `Trainer.load_checkpoint()` resumes from update 124848, so `skipped_epoch` is far above `epochs` and the training loop never runs. The old LR schedule would also override `--learning_rate` | Step 6 creates an **EMA-weights-only** checkpoint, so training starts at update 0 with a fresh optimizer and our LR |
| 2 | `--batch_size_per_gpu 800` | `DynamicBatchSampler` **silently discards** any clip longer than the frame budget. 800 frames is about 8.5 s, so more than half of the clips would never be used | Budget of **2400 frames**. All clips are ≤ 20 s (1875 frames) after preprocessing |
| 3 | Checkpoints were written to the Colab local disk | `finetune_cli` saves to `f5_tts/../../ckpts/<dataset_name>`. With an absolute dataset path, that ends up outside Drive and is lost on disconnect | The patched script saves to **Drive** (`ckpts/kathbath_malayalam_ft`), so training can be resumed |
| 4 | `UnboundLocalError: preprocessed_mel` | Upstream bug in `load_dataset(..., "CustomDatasetPath")` | The patched script builds the `CustomDataset` directly. **No editing of site-packages** |
| 5 | DataLoader warning: 16 workers on a 2-CPU machine | `trainer.train()` defaults to `num_workers=16` | Patched to `num_workers=2` |

### Drive layout used
```
/content/drive/MyDrive/Datsets/F5-TTS-Malayalam/
├── dataset/vocab.txt                              # from notebook 2
├── dataset/kathbath_malayalam_finetune/           # from notebook 2
├── checkpoints/model_last.pt                      # 5 GB original HF checkpoint (from the first run; optional)
├── ckpts/kathbath_malayalam_ft/                   # ← fine-tuning checkpoints (new)
│   ├── pretrained_openbible_malayalam_ema.pt      #   EMA-only start checkpoint (≈1.3 GB)
│   └── model_last.pt                              #   latest fine-tune state (≈5 GB, overwritten)
├── runs/F5TTS_v1_Base/                            # TensorBoard logs
├── logs/train_*.log                               # full training console logs
└── exports/f5tts_malayalam_kathbath_ft.pt         # inference-only weights (≈1.3 GB)
```
> **Drive space:** you need about **8 GB free**. You can delete the old `checkpoints/model_last.pt` (5 GB) once step 6 has created the EMA checkpoint.

## 1. Mount Google Drive and define every path

**What we do:** mount Drive, define all paths in one place and confirm that the outputs of notebook 2 exist. This cell also defines `run_in_subprocess()`, which the checkpoint steps (6, 7, 12) use to keep this notebook's memory small (see step 6).

**What happened:** project, prepared dataset and vocab all exist on Drive.

In [ ]:
import subprocess
import sys
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/Datsets/F5-TTS-Malayalam")

# Inputs (created by the preprocessing notebook)
PREPARED_DIR = PROJECT_DIR / "dataset" / "kathbath_malayalam_finetune"
VOCAB_PATH   = PROJECT_DIR / "dataset" / "vocab.txt"

# Pretrained model (pinned to the exact snapshot used so far)
MODEL_ID       = "multilingual-tts/F5-TTS-OpenBible-Malayalam"
MODEL_REVISION = "fa4ad6e0d147ac4ab6fdc9e783ed10fbabcde6d3"
OLD_FULL_CHECKPOINT = PROJECT_DIR / "checkpoints" / "model_last.pt"   # from the first run, if present

# Fine-tuning outputs (all on Drive)
CKPT_OUT_DIR   = PROJECT_DIR / "ckpts" / "kathbath_malayalam_ft"
PRETRAINED_EMA = CKPT_OUT_DIR / "pretrained_openbible_malayalam_ema.pt"
SCRIPTS_DIR    = PROJECT_DIR / "scripts"
PATCHED_SCRIPT = SCRIPTS_DIR / "finetune_malayalam.py"
LOG_DIR        = PROJECT_DIR / "logs"
TB_DIR         = PROJECT_DIR / "runs"
EXPORT_DIR     = PROJECT_DIR / "exports"
EXPORT_PATH    = EXPORT_DIR / "f5tts_malayalam_kathbath_ft.pt"

for folder in (CKPT_OUT_DIR, SCRIPTS_DIR, LOG_DIR, TB_DIR, EXPORT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

for name, path in {
    "Prepared dataset": PREPARED_DIR,
    "raw.arrow": PREPARED_DIR / "raw.arrow",
    "duration.json": PREPARED_DIR / "duration.json",
    "Model vocab": VOCAB_PATH,
}.items():
    print(f"{name:17}: {path.exists()}  {path}")
    assert path.exists(), f"{name} missing, so run the preprocessing notebook first."

print("\nCheckpoints will be saved to:", CKPT_OUT_DIR)


def run_in_subprocess(code: str, *args) -> None:
    """Run memory-heavy checkpoint code in a separate Python process.

    Python rarely returns freed RAM to the OS. If a 5 GB checkpoint were loaded
    in this notebook, the kernel would keep that memory while training runs,
    and Colab (12.7 GB RAM) would crash. A subprocess releases everything when it exits.
    """
    result = subprocess.run([sys.executable, "-c", code, *map(str, args)], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr[-3000:])
        raise RuntimeError(f"Subprocess failed with return code {result.returncode}")

## 2. Install the pinned libraries

**What we do:** install exactly the same stack as the preprocessing notebook: PyTorch 2.8.0/CUDA 12.8 first, then `f5-tts==1.1.22` + `transformers==5.16.1`. This notebook previously installed an unpinned `f5-tts`. It is now pinned, because every patch below depends on the 1.1.22 source.

**What happened:** installed without errors.

> If Colab shows a *"Restart session"* prompt, restart, re-run **step 1**, then continue from **step 3**.

In [ ]:
!pip install -q \
    torch==2.8.0+cu128 \
    torchaudio==2.8.0+cu128 \
    torchvision==0.23.0+cu128 \
    --index-url https://download.pytorch.org/whl/cu128

!pip install -q \
    f5-tts==1.1.22 \
    transformers==5.16.1

## 3. Verify the environment, GPU and RAM

**What we do:** print the versions (and stop if any differ from the tested set), the GPU, VRAM and system RAM, plus the free space on Drive.

**What happened:** Python 3.13.15, PyTorch 2.8.0+cu128, Torchaudio 2.8.0+cu128, Torchvision 0.23.0+cu128, Transformers 5.16.1, F5-TTS 1.1.22, CUDA 12.8, **Tesla T4 with 14.56 GB**, 12.67 GB RAM.

> The T4 has no native bfloat16 support, so training uses **fp16** mixed precision.

In [ ]:
import sys
import shutil
import importlib.metadata

import psutil
import torch
import torchaudio
import torchvision
import transformers

INSTALLED = {
    "torch": torch.__version__,
    "torchaudio": torchaudio.__version__,
    "torchvision": torchvision.__version__,
    "transformers": transformers.__version__,
    "f5-tts": importlib.metadata.version("f5-tts"),
}
EXPECTED = {
    "torch": "2.8.0+cu128",
    "torchaudio": "2.8.0+cu128",
    "torchvision": "0.23.0+cu128",
    "transformers": "5.16.1",
    "f5-tts": "1.1.22",
}

print("Python      :", sys.version.split()[0])
for name, version in INSTALLED.items():
    print(f"{name:12}: {version}")
print("CUDA        :", torch.version.cuda)

assert torch.cuda.is_available(), "No GPU, so select Runtime > Change runtime type > T4 GPU."
print("GPU         :", torch.cuda.get_device_name(0))
print("GPU memory  :", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
print("System RAM  :", round(psutil.virtual_memory().total / 1024**3, 2), "GB")
print("Drive free  :", round(shutil.disk_usage("/content/drive").free / 1024**3, 1), "GB (as reported by the Drive mount)")

mismatch = {k: (INSTALLED[k], v) for k, v in EXPECTED.items() if INSTALLED[k] != v}
assert not mismatch, f"Version mismatch (installed, expected): {mismatch}"
print("\nAll versions match the tested environment.")

## 4. Read the pretrained model configuration

**What we do:** download the model's YAML config (pinned revision) and print the parts that must match our training setup.

**What happened:** `F5TTS_v1_Base`, backbone `DiT`, `dim 1024 / depth 22 / heads 16 / ff_mult 2 / text_dim 512 / conv_layers 4`, `text_mask_padding True`, `pe_attn_head null`, tokenizer `custom`, mel `vocos`. These are the **same values** `finetune_cli --exp_name F5TTS_v1_Base` builds, so the checkpoint loads without architecture changes.

In [ ]:
from huggingface_hub import hf_hub_download
from omegaconf import OmegaConf

CONFIG_PATH = hf_hub_download(
    repo_id=MODEL_ID,
    filename="F5-TTS_OpenBible_Malayalam.yaml",
    revision=MODEL_REVISION,
)
model_yaml = OmegaConf.load(CONFIG_PATH)

print("Config    :", CONFIG_PATH)
print("Model     :", model_yaml.model.name)
print("Backbone  :", model_yaml.model.backbone)
print("Tokenizer :", model_yaml.model.tokenizer)
print("Mel type  :", model_yaml.model.mel_spec.mel_spec_type)
print("Arch      :", OmegaConf.to_container(model_yaml.model.arch))

## 5. Build the patched fine-tuning script

**What we do:** copy `f5_tts.train.finetune_cli` from the installed 1.1.22 package and apply **3 patches**. The result is saved to `scripts/finetune_malayalam.py`:

1. **Checkpoint folder → Drive.** `checkpoint_path` becomes `ckpts/kathbath_malayalam_ft` on Drive, instead of the local disk.
2. **Dataset loading.** The upstream `load_dataset(..., "CustomDatasetPath")` crashes with `UnboundLocalError: preprocessed_mel`. Instead, the script builds `CustomDataset(raw.arrow, durations, preprocessed_mel=False, ...)` directly. This is exactly what the upstream code intends, with nothing edited in `site-packages`, so there is nothing to re-patch after a runtime restart.
3. **DataLoader workers.** `trainer.train(..., num_workers=2)` replaces the default of 16, which is too many for Colab's 2 CPUs.

The script is rebuilt from the installed package every time. The cell stops if an expected line is not found, which would mean the f5-tts version changed.

**What happened in the first version:** only the dataset line was patched, and the run failed with `UnboundLocalError`. The fix was then applied by editing `site-packages/f5_tts/model/dataset.py`, which is lost on every new runtime. That approach is replaced by patch 2.

In [ ]:
import inspect
from f5_tts.train import finetune_cli

source = inspect.getsource(finetune_cli)

PATCHES = [
    (
        "checkpoint folder on Google Drive",
        '    checkpoint_path = str(files("f5_tts").joinpath(f"../../ckpts/{args.dataset_name}"))',
        f'    checkpoint_path = {str(CKPT_OUT_DIR)!r}  # Malayalam patch: checkpoints on Google Drive',
    ),
    (
        "load prepared raw.arrow directly",
        '    train_dataset = load_dataset(args.dataset_name, tokenizer, mel_spec_kwargs=mel_spec_kwargs)',
        '''    # Malayalam patch: load the prepared dataset folder directly.
    # (f5-tts 1.1.22 load_dataset(..., "CustomDatasetPath") raises UnboundLocalError: preprocessed_mel)
    import json
    from datasets import Dataset as Dataset_
    from f5_tts.model.dataset import CustomDataset

    with open(f"{args.dataset_name}/duration.json", "r", encoding="utf-8") as f:
        durations = json.load(f)["duration"]
    train_dataset = CustomDataset(
        Dataset_.from_file(f"{args.dataset_name}/raw.arrow"),
        durations=durations,
        preprocessed_mel=False,
        **mel_spec_kwargs,
    )
    print(f"Loaded {len(train_dataset)} samples from {args.dataset_name}")''',
    ),
    (
        "DataLoader workers = 2",
        '''    trainer.train(
        train_dataset,
        resumable_with_seed=666,  # seed for shuffling dataset
    )''',
        '''    trainer.train(
        train_dataset,
        num_workers=2,  # Malayalam patch: Colab has 2 CPUs (default 16)
        resumable_with_seed=666,  # seed for shuffling dataset
    )''',
    ),
]

for name, old, new in PATCHES:
    if source.count(old) != 1:
        raise RuntimeError(f"Patch '{name}': expected code not found. Is f5-tts still 1.1.22?")
    source = source.replace(old, new)
    print("Patched:", name)

PATCHED_SCRIPT.write_text(source, encoding="utf-8")
print("\nWritten:", PATCHED_SCRIPT)

## 6. Create the EMA-only starting checkpoint (fixes "no training happened")

**What we do:**
1. Load the original 5 GB checkpoint. It is reused from `checkpoints/model_last.pt` on Drive if the first run saved it there, otherwise it is downloaded from Hugging Face (pinned revision).
2. Print its keys and `update` value to show why it cannot be used directly.
3. Save **only `ema_model_state_dict`** as `pretrained_openbible_malayalam_ema.pt` inside the checkpoint folder.

**Why this works (f5-tts 1.1.22 `Trainer.load_checkpoint`):**
- With a checkpoint that contains `update`, the trainer restores the step count, optimizer and LR scheduler and **continues from update 124848**. That is what happened before: `Saved last checkpoint at update 124848` and zero training steps.
- With a checkpoint that has **only** `ema_model_state_dict` (the same format as the official F5-TTS release checkpoints), the trainer loads those weights into both the model and the EMA and **starts at update 0**, with a fresh optimizer, our learning rate and our warmup.

The file name starts with `pretrained_` and sits inside the checkpoint folder, so `finetune_cli` finds it and does not copy it again.

**Memory:** the 5 GB checkpoint is loaded in a **separate process** (`run_in_subprocess`), so its RAM is released when the process ends. In the previous version it was loaded in the notebook itself. The kernel kept that memory, and the Colab session **crashed from running out of RAM** about 7 minutes into training.

**What happened in the first run:** the checkpoint keys were `model_state_dict, optimizer_state_dict, ema_model_state_dict, scheduler_state_dict, update`, with a size of 5.01 GB.
**What happened in the second run:** the EMA-only file was created, and training then **started at update 0**, which confirms this fix.
This runs once. Later runs skip it because the file already exists on Drive.

In [ ]:
from huggingface_hub import hf_hub_download

EXTRACT_EMA_CODE = '''
import sys
import torch

source_ckpt, target = sys.argv[1], sys.argv[2]
print("Loading full checkpoint:", source_ckpt)
full = torch.load(source_ckpt, map_location="cpu", weights_only=True)
print("Keys  :", list(full.keys()))
print("update:", full.get("update"), "<- the trainer would resume from here")
torch.save({"ema_model_state_dict": full["ema_model_state_dict"]}, target)
print("Saved EMA-only checkpoint:", target)
'''

if PRETRAINED_EMA.exists():
    print("EMA-only checkpoint already exists, skipping:", PRETRAINED_EMA)
else:
    if OLD_FULL_CHECKPOINT.exists():
        source_ckpt = OLD_FULL_CHECKPOINT
    else:
        source_ckpt = hf_hub_download(
            repo_id=MODEL_ID,
            filename="model_last.pt",
            revision=MODEL_REVISION,
        )
    run_in_subprocess(EXTRACT_EMA_CODE, source_ckpt, PRETRAINED_EMA)

print("EMA checkpoint:", PRETRAINED_EMA)
print("Size          :", round(PRETRAINED_EMA.stat().st_size / 1024**3, 2), "GB")

## 7. Check that the model, vocab and checkpoint fit together

**What we do:** build the model exactly as the patched script will (`F5TTS_v1_Base` DiT + CFM, 24 kHz vocos mel, custom tokenizer from our `vocab.txt`). Then load the EMA weights exactly as `Trainer.load_checkpoint` does, with `strict=True`, so any shape or key mismatch fails here instead of 10 minutes into training.

Like step 6, this runs in a separate process so the model copy does not stay in the notebook's RAM.

**What happened in the first run (same check with `model_state_dict`):** tokenizer vocab size 119, highest index 118, `text_embed` weight `(120, 512)` (119 characters + 1 filler token), and `Model + checkpoint compatibility: OK`.

In [ ]:
CHECK_MODEL_CODE = '''
import sys
import torch
from f5_tts.model import CFM, DiT
from f5_tts.model.utils import get_tokenizer

vocab_path, ema_path = sys.argv[1], sys.argv[2]

vocab_char_map, vocab_size = get_tokenizer(vocab_path, "custom")
print("Tokenizer vocab size:", vocab_size, "| highest index:", max(vocab_char_map.values()))

# Same values as finetune_cli.py for --exp_name F5TTS_v1_Base (and the model YAML)
model = CFM(
    transformer=DiT(
        dim=1024, depth=22, heads=16, ff_mult=2, text_dim=512, conv_layers=4,
        text_num_embeds=vocab_size, mel_dim=100,
    ),
    mel_spec_kwargs=dict(
        n_fft=1024, hop_length=256, win_length=1024,
        n_mel_channels=100, target_sample_rate=24000, mel_spec_type="vocos",
    ),
    vocab_char_map=vocab_char_map,
)

ckpt = torch.load(ema_path, map_location="cpu", weights_only=True)
state = {
    k.replace("ema_model.", ""): v
    for k, v in ckpt["ema_model_state_dict"].items()
    if k not in ["initted", "update", "step"]
}
for key in ["mel_spec.mel_stft.mel_scale.fb", "mel_spec.mel_stft.spectrogram.window"]:
    state.pop(key, None)

model.load_state_dict(state, strict=True)

emb_shape = tuple(state["transformer.text_embed.text_embed.weight"].shape)
print("text_embed weight   :", emb_shape)
assert emb_shape == (vocab_size + 1, 512)
print("Parameters          :", round(sum(p.numel() for p in model.parameters()) / 1e6, 1), "M")
print("Model + vocab + EMA checkpoint compatibility: OK")
'''

run_in_subprocess(CHECK_MODEL_CODE, VOCAB_PATH, PRETRAINED_EMA)

## 8. Training configuration

**What we do:** set the hyper-parameters. The table explains each choice. Frames are mel frames: 1 s ≈ 93.75 frames (24 000 Hz / hop 256).

| Setting | Value | Reason |
|---|---|---|
| `learning_rate` | `1e-5` | Low LR for fine-tuning an already-trained Malayalam model (kept from your setup) |
| `batch_size_per_gpu` (frame) | `2400` | ≈ 25.6 s of audio per batch. Must be ≥ 1875 (the longest clip is 20 s), otherwise clips are **silently dropped** |
| `max_samples` | `8` | Upper limit on clips per batch. With about 10 s clips a batch holds 2–5 clips |
| `grad_accumulation_steps` | `4` | Effective batch ≈ 9600 frames (≈ 100 s of audio) per update |
| `epochs` | `10` | **Measured: 233 updates per epoch** (1761 clips) → **4660 updates** in total |
| `num_warmup_updates` | `200` | About 5 % warmup, then linear decay |
| `keep_last_n_checkpoints` | `0` | No numbered `model_<N>.pt` files. Each full checkpoint is about 5 GB and would fill Drive |
| `last_per_updates` | `250` | Overwrites `model_last.pt` on Drive about every epoch, which is enough to resume |
| mixed precision | `fp16` | T4 supports fp16 tensor cores |

**Training time (measured on T4):** about 6–7 s per update, so about **25 min per epoch** and **about 8 h for 20 epochs**. That is longer than a typical Colab session, but `model_last.pt` is saved every 250 updates (about 26 min), so a disconnect costs at most about 26 min of work. To finish in one session, set `--epochs` to `8` (about 3.5 h) **before the first start**.

**If you get `CUDA out of memory`:** set `batch_size_per_gpu` to `2000`, and optionally `max_samples` to `4`. **Never go below 1900** unless you also lower `MAX_DURATION_SEC` in the preprocessing notebook.

**Resuming:** re-run the notebook with the **same** settings. The trainer finds `model_last.pt` in the checkpoint folder and continues from there. Changing `epochs` or the batch settings while resuming breaks the LR schedule.

`START_FRESH = True` deletes previous fine-tune checkpoints (`model_*.pt`) in `ckpts/kathbath_malayalam_ft`. The EMA start checkpoint is kept.

In [ ]:
START_FRESH = False   # True = delete previous fine-tune checkpoints and start again from update 0

TRAIN_ARGS = {
    "--learning_rate": "1e-5",
    "--batch_size_per_gpu": "2400",
    "--batch_size_type": "frame",
    "--max_samples": "8",
    "--grad_accumulation_steps": "4",
    "--max_grad_norm": "1.0",
    "--epochs": "10",
    "--num_warmup_updates": "200",
    "--save_per_updates": "1000",
    "--keep_last_n_checkpoints": "0",
    "--last_per_updates": "250",
}

previous = sorted(CKPT_OUT_DIR.glob("model_*.pt"))
if previous and START_FRESH:
    for p in previous:
        p.unlink()
        print("Deleted:", p.name)
    previous = []

print("Checkpoint folder:", CKPT_OUT_DIR)
print("Contents         :", sorted(p.name for p in CKPT_OUT_DIR.iterdir()))
print("Mode             :", "RESUME from model_last.pt" if previous else "FRESH start from the EMA checkpoint (update 0)")

## 9. Build and sanity-check the launch command

**What we do:** assemble the `accelerate launch` command. All `accelerate` options are passed explicitly (fp16, 1 process, 1 machine, no dynamo), which also removes the "values were not passed" warning. Then run the patched script with `--help` as a quick syntax check.

**What happened in the first version:** `accelerate test` passed on the T4, and the patched script's `--help` returned code 0.

In [ ]:
import subprocess
import sys

accelerate_cmd = [
    "accelerate", "launch",
    "--mixed_precision", "fp16",
    "--num_processes", "1",
    "--num_machines", "1",
    "--dynamo_backend", "no",
    str(PATCHED_SCRIPT),
    "--exp_name", "F5TTS_v1_Base",
    "--dataset_name", str(PREPARED_DIR),
    *[item for pair in TRAIN_ARGS.items() for item in pair],
    "--finetune",
    "--pretrain", str(PRETRAINED_EMA),
    "--tokenizer", "custom",
    "--tokenizer_path", str(VOCAB_PATH),
    "--logger", "tensorboard",
]

check = subprocess.run([sys.executable, str(PATCHED_SCRIPT), "--help"], capture_output=True, text=True)
assert check.returncode == 0, check.stderr
print("Patched script --help: OK\n")

print("========== COMMAND ==========")
print(" \\\n  ".join(accelerate_cmd))

## 10. Start TensorBoard (optional, run before training)

**What we do:** open TensorBoard on the Drive log folder. The trainer writes its logs to `runs/F5TTS_v1_Base` relative to the working directory, and training runs with `cwd = PROJECT_DIR`, so the logs persist on Drive. Watch `loss` go down and `lr` warm up to `1e-5`.

It may show "No dashboards" until the first steps have been logged. Use the refresh button.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {str(TB_DIR)}

## 11. Run fine-tuning

**What we do:** launch training as a subprocess and **stream its output live** into this cell. Every line is also written to `logs/train_<timestamp>.log` on Drive.
- `TQDM_MININTERVAL=30` limits progress-bar updates to one every 30 s so the output stays readable.
- **About once a minute a `[RAM]` line is printed** with system memory use. If it gets close to the total (12.7 GB), stop the cell before Colab crashes, and tell me the numbers.
- The torchaudio `UserWarning`s about *TorchCodec* / *StreamingMediaDecoder* are harmless deprecation notices from torchaudio 2.8, and they are now filtered out.

**What to look for:**
- `vocab :  119`, `vocoder :  vocos`, `Loaded 1761 samples ...`
- a progress bar `Epoch 1/20 ... /233` with an `update` count that **starts at 0**, not 124848
- `Saved last checkpoint at update 250`, `500`, ... on Drive

**What happened:**
- First version: return code 0 but `Saved last checkpoint at update 124848`, with **no steps trained**. Fixed in step 6.
- Second version: **training worked.** It loaded 1761 samples, created 233 updates per epoch and started at update 1, with loss 0.9–1.9 at about 6–7 s per update. At update 21 the Colab session **crashed from running out of RAM**, because the notebook kernel still held the 5 GB checkpoint from steps 6–7. Steps 6, 7 and 12 now run in separate processes.

**Resuming after a disconnect:** reconnect and run steps 1–3 and 5, then 8–11. Steps 6 and 7 are optional on a resume. Training continues from the last `model_last.pt`.

In [ ]:
import os
import time
import datetime
import subprocess

import psutil

log_path = LOG_DIR / f"train_{datetime.datetime.now():%Y%m%d_%H%M%S}.log"

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["TQDM_MININTERVAL"] = "30"
env["PYTHONWARNINGS"] = (
    "ignore::UserWarning:torchaudio._backend.utils,"
    "ignore::UserWarning:torchaudio._backend.ffmpeg"
)

vm = psutil.virtual_memory()
print(f"RAM before training: {vm.used / 1024**3:.1f} / {vm.total / 1024**3:.1f} GB used")
print("Log file:", log_path, "\n")

with open(log_path, "w", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        accelerate_cmd,
        cwd=str(PROJECT_DIR),          # TensorBoard logs -> PROJECT_DIR/runs
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    next_ram_report = time.time() + 60
    for line in process.stdout:
        if time.time() >= next_ram_report:
            vm = psutil.virtual_memory()
            line = f"[RAM] {vm.used / 1024**3:.1f} / {vm.total / 1024**3:.1f} GB used\n" + line
            next_ram_report = time.time() + 60
        print(line, end="")
        log_file.write(line)
        log_file.flush()
    return_code = process.wait()

print("\nReturn code:", return_code)
if return_code != 0:
    raise RuntimeError(f"Training failed, see the output above or {log_path}")

## 12. Verify training really happened and export inference weights

**What we do:**
1. Load `model_last.pt` from the checkpoint folder and confirm that `update` is a **small positive number** (the fine-tune steps), not 124848.
2. Export only `model_state_dict` (the fine-tuned weights, about 1.3 GB) to `exports/`, for inference.

**Why not the EMA weights?** The trainer updates EMA only every 10 steps, with decay 0.9999. After a few thousand fine-tune updates the EMA copy is still about 94 % the original model. The F5-TTS docs note: *"`use_ema = True` might be harmful for early-stage finetuned checkpoints"*. So the raw fine-tuned weights are exported and used.

The 5 GB checkpoint is loaded in a separate process, like step 6, so the notebook RAM stays free for step 13.

**Expected result:** `update` = **4660** after all 20 epochs, or less if you run this after an intermediate save. The export is about 1.3 GB.

In [ ]:
EXPORT_CODE = '''
import sys
import torch

last_ckpt, export_path = sys.argv[1], sys.argv[2]
ckpt = torch.load(last_ckpt, map_location="cpu", weights_only=True)
print("Checkpoint:", last_ckpt)
print("Keys      :", list(ckpt.keys()))
print("update    :", ckpt["update"])
assert 0 < ckpt["update"] < 124848, "Unexpected update value, so this does not look like a fresh fine-tune."
torch.save({"model_state_dict": ckpt["model_state_dict"]}, export_path)
print("Exported inference weights:", export_path)
'''

last_ckpt = CKPT_OUT_DIR / "model_last.pt"
assert last_ckpt.exists(), "No model_last.pt, so training has not saved a checkpoint yet."

run_in_subprocess(EXPORT_CODE, last_ckpt, EXPORT_PATH)
print("Size:", round(EXPORT_PATH.stat().st_size / 1024**3, 2), "GB")

## 13. Listen to the fine-tuned model

**What we do:** load the exported weights with the standard F5-TTS inference helpers (`use_ema=False`, see step 12) and the vocos vocoder. We take a 5–9 s clip from the training data as the voice reference and synthesise a new Malayalam sentence.

F5-TTS is zero-shot: the **reference clip decides the voice**. Try your own clean 5–10 s recording with its exact transcript to hear how the model generalises.

**Expected result:** a 24 kHz WAV in the reference speaker's voice, saved to `exports/test_output.wav`.

In [ ]:
import soundfile as sf
from datasets import Dataset
from hydra.utils import get_class
from IPython.display import Audio, display

from f5_tts.infer.utils_infer import (
    infer_process,
    load_model,
    load_vocoder,
    preprocess_ref_audio_text,
)

device = "cuda"

model_cls = get_class(f"f5_tts.model.{model_yaml.model.backbone}")
vocoder = load_vocoder(vocoder_name="vocos", is_local=False, device=device)
ft_model = load_model(
    model_cls,
    model_yaml.model.arch,
    str(EXPORT_PATH),
    mel_spec_type="vocos",
    vocab_file=str(VOCAB_PATH),
    use_ema=False,
    device=device,
)

# Reference voice: first training clip between 5 and 9 seconds
raw = Dataset.from_file(str(PREPARED_DIR / "raw.arrow"))
ref = next(row for row in raw if 5.0 <= row["duration"] <= 9.0)
print("Reference audio:", ref["audio_path"])
print("Reference text :", ref["text"])
display(Audio(ref["audio_path"]))

gen_text = "കേരളം ഇന്ത്യയുടെ തെക്കുപടിഞ്ഞാറൻ തീരത്തുള്ള ഒരു സംസ്ഥാനമാണ്."
vocab_chars = set(VOCAB_PATH.read_text(encoding="utf-8").splitlines())
assert not (set(gen_text) - vocab_chars), f"Characters not in vocab: {set(gen_text) - vocab_chars}"

ref_audio, ref_text = preprocess_ref_audio_text(ref["audio_path"], ref["text"])
wave, sample_rate, _ = infer_process(
    ref_audio, ref_text, gen_text, ft_model, vocoder,
    mel_spec_type="vocos", device=device,
)

out_path = EXPORT_DIR / "test_output.wav"
sf.write(str(out_path), wave, sample_rate)
print("\nGenerated:", out_path, "| sample rate:", sample_rate)
display(Audio(str(out_path)))

## ✅ Done

- Fine-tune checkpoints: `ckpts/kathbath_malayalam_ft/model_last.pt` (resumable)
- Inference weights: `exports/f5tts_malayalam_kathbath_ft.pt`. Use it with `use_ema=False`, `vocab.txt` and the model YAML, exactly as in step 13.
- Logs: `runs/` (TensorBoard) and `logs/` (console)

**Quality notes**
- Kathbath audio is 16 kHz, so compare against the base model on the same reference clip to judge whether fine-tuning helped.
- If the output is worse than the base model, lower `--epochs` or the learning rate. With only about 5 h of data, overfitting is possible.